In [7]:
import os
import cv2
import numpy as np
import shutil
import random
from pathlib import Path

In [8]:
# Input and output directories
input_folder = "videos without polyps"
images_folder = "images"
masks_folder = "masks"

In [9]:
# Create output folders if they don't exist
os.makedirs(images_folder, exist_ok=True)
os.makedirs(masks_folder, exist_ok=True)

In [10]:
# Frame sampling step
frame_step = 250

In [11]:
total_videos = 0
total_sampled_frames = 0

# Process each AVI file in the input folder
for video_file in os.listdir(input_folder):
    if video_file.lower().endswith(".avi"):
        
        total_videos += 1
        
        video_path = os.path.join(input_folder, video_file)
        video_name = os.path.splitext(video_file)[0]
        
        cap = cv2.VideoCapture(video_path)
        
        if not cap.isOpened():
            print(f"Error opening video file: {video_file}")
            continue
        
        frame_index = 0
        saved_frame_count = 0
        
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # Sample every 50 frames
            if frame_index % frame_step == 0:
                
                saved_frame_count += 1
                frame_number_str = f"{saved_frame_count:04d}"
                
                # Save frame
                image_filename = f"{video_name}_{frame_number_str}.png"
                image_path = os.path.join(images_folder, image_filename)
                cv2.imwrite(image_path, frame)
                
                # Create total black mask (same height and width)
                height, width = frame.shape[:2]
                black_mask = np.zeros((height, width), dtype=np.uint8)
                
                # Save mask
                mask_filename = f"{video_name}_{frame_number_str}_mask.png"
                mask_path = os.path.join(masks_folder, mask_filename)
                cv2.imwrite(mask_path, black_mask)
            
            frame_index += 1
        
        cap.release()
        
        total_sampled_frames += saved_frame_count
        
        print(f"Video: {video_file}")
        print(f" → Total frames sampled: {saved_frame_count}")
        print("-" * 40)

print("Processing complete.")
print(f"Total videos processed: {total_videos}")
print(f"Total sampled frames across all videos: {total_sampled_frames}")

Video: 0_50.avi
 → Total frames sampled: 37
----------------------------------------
Video: 0_44.avi
 → Total frames sampled: 37
----------------------------------------
Video: 0_45.avi
 → Total frames sampled: 53
----------------------------------------
Video: 0_51.avi
 → Total frames sampled: 37
----------------------------------------
Video: 0_47.avi
 → Total frames sampled: 24
----------------------------------------
Video: 0_53.avi
 → Total frames sampled: 41
----------------------------------------
Video: 0_52.avi
 → Total frames sampled: 41
----------------------------------------
Video: 0_46.avi
 → Total frames sampled: 41
----------------------------------------
Video: 0_42.avi
 → Total frames sampled: 32
----------------------------------------
Video: 0_56.avi
 → Total frames sampled: 33
----------------------------------------
Video: 0_57.avi
 → Total frames sampled: 25
----------------------------------------
Video: 0_43.avi
 → Total frames sampled: 49
---------------------

In [12]:
# ---- CONFIG ----
images_dir = Path("images")
masks_dir = Path("masks")

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random.seed(42)

Older split (0.8 - 0.1 - 0.1)

Total pairs: 2042
Train: 1633
Val: 204
Test: 205

In [13]:
# Create output folders
for split in ["train", "val", "test"]:
    (Path(split) / "images").mkdir(parents=True, exist_ok=True)
    (Path(split) / "masks").mkdir(parents=True, exist_ok=True)

# Collect image-mask pairs
pairs = []

for img_path in images_dir.iterdir():
    if img_path.is_file():
        mask_name = img_path.stem + "_mask" + img_path.suffix
        mask_path = masks_dir / mask_name

        if mask_path.exists():
            pairs.append((img_path, mask_path))

# Shuffle pairs
random.shuffle(pairs)

# Split
total = len(pairs)
train_end = int(total * train_ratio)
val_end = train_end + int(total * val_ratio)

splits = {
    "train": pairs[:train_end],
    "val": pairs[train_end:val_end],
    "test": pairs[val_end:]
}

# Copy files
for split_name, split_pairs in splits.items():
    for img_path, mask_path in split_pairs:
        shutil.copy2(img_path, Path(split_name) / "images" / img_path.name)
        shutil.copy2(mask_path, Path(split_name) / "masks" / mask_path.name)

print(f"Total pairs: {total}")
print(f"Train: {len(splits['train'])}")
print(f"Val: {len(splits['val'])}")
print(f"Test: {len(splits['test'])}")

Total pairs: 2042
Train: 1429
Val: 204
Test: 409
